# Poker Solver — GPU Benchmark

1. `Runtime -> Change runtime type -> GPU -> Save`
2. Upload `poker_solver_gpu.zip` via the **left sidebar folder icon -> Upload**.
3. `Runtime -> Run all`.

Three timing tables: cell 6 GPU float64, cell 6b GPU float32, cell 7 CPU.


In [ ]:
# 1) Confirm a GPU is attached
!nvidia-smi -L || echo 'NO GPU — Runtime -> Change runtime type -> GPU'

In [ ]:
# 2) Unpack the code (upload poker_solver_gpu.zip via the left sidebar first)
import zipfile, os, glob
hits = glob.glob('/content/poker_solver_gpu.zip') or glob.glob('/content/*.zip')
if not hits:
    raise FileNotFoundError('Upload poker_solver_gpu.zip via the left sidebar folder icon, then re-run.')
with zipfile.ZipFile(hits[0]) as z:
    z.extractall('poker')
print('unpacked ->', sorted(os.listdir('poker')))

In [ ]:
# 3) Verify CuPy sees the GPU (self-heals with a pip install if needed)
try:
    import cupy as cp
except Exception:
    import subprocess, sys
    subprocess.run([sys.executable,'-m','pip','install','-q','cupy-cuda12x'])
    import cupy as cp
name = cp.cuda.runtime.getDeviceProperties(0)['name']
if isinstance(name, bytes): name = name.decode()
print('cupy', cp.__version__, '| device:', name)

In [ ]:
# 4) Correctness: GPU backend must match CPU backend
import numpy as np, sys
sys.path.insert(0, 'poker/src')
from pokertrainer.cards import parse_cards, parse_hand
from pokertrainer.solver.batched_gpu import BatchedGPUCFR
flop = parse_cards('As7h2d')
oop = [parse_hand(h) for h in ['AhAc','KsKc','7s7c','AhKh','Ts9s']]
ip  = [parse_hand(h) for h in ['AhQh','KsKh','JsJh','AhTh','Tc9c']]
wo, wi = np.ones(5), np.ones(5)
g = BatchedGPUCFR(flop,oop,ip,wo,wi,5.5,0.66,streets=2,backend='cupy').run(200)
c = BatchedGPUCFR(flop,oop,ip,wo,wi,5.5,0.66,streets=2,backend='numpy').run(200)
diff = abs(g['root_ev_oop_bb']-c['root_ev_oop_bb'])
print('GPU EV %.6f%%  |  CPU EV %.6f%%  |  diff %.2e' % (g['root_ev_pct_pot'], c['root_ev_pct_pot'], diff))
print('PASS (GPU==CPU)' if diff < 1e-6 else 'CHECK')

## The benchmark — three tables

`streets=3` is the full river solve. `ms/iter × 600 / 1000` ≈ seconds/board; `× 12 / 60` ≈ minutes for a 12-board library.


In [ ]:
# 6) GPU float64 (exact)
%env POKER_BACKEND=cupy
%env POKER_DTYPE=float64
!cd poker && PYTHONPATH=src python bench/gpu_bench.py 3 80 160 250

In [ ]:
# 6b) GPU float32 (consumer GPUs are FP64-crippled — usually MUCH faster; accuracy cost ~1e-6 of pot)
%env POKER_BACKEND=cupy
%env POKER_DTYPE=float32
!cd poker && PYTHONPATH=src python bench/gpu_bench.py 3 80 160 250

In [ ]:
# 7) CPU baseline (smaller sizes — CPU is slow at streets=3)
%env POKER_BACKEND=numpy
%env POKER_DTYPE=float64
!cd poker && PYTHONPATH=src python bench/gpu_bench.py 3 40 80

## Reading the result

Compare **ms/iter** at 250 combos: cell 6 (GPU f64), cell 6b (GPU f32), cell 7 (CPU).

- **Go:** fastest GPU row at ~250 combos in the tens-to-low-hundreds of ms → minutes/board → practical, fully permissive. An A100/H100 adds ~10–30× over a T4.
- **Marginal:** still seconds/iter in float32 → needs isomorphism / range abstraction or a bigger GPU.
